# Anonymisation (step 10)

Produces the releasable tables at each de-identification level from the harmonised files and the static table. Everything runs inside the SDE. A level is a `Config` in `anonymise.py` (`ANONYMISATION.md` describes the levels, the steps and the checks). A failed check stops the run and releases nothing; a warning does not. Reports hold counts only, and any figure must pass the output check before it leaves the SDE.

Sections: 1 setup, 2 Tier 1, 3 Tier 2, 4 what the run changed, 5 diagnostics (only when a check warns or fails), 6 text vocabularies, 7 checks on the released files, 8 figures, 9 other levels.

In [ ]:
%load_ext autoreload
%autoreload 2
import sys
from pathlib import Path
import pandas as pd

open_picu_dir = Path("/home/jormond/Documents/DS217/users/jormond/open_picu")
sys.path.insert(0, str(open_picu_dir / "code"))

import anonymise as AN
import reid as R

## 1. Setup

Directories, the check that the right module version is loaded, and the pooled static table (all sites, one row per stay). `run` applies the cohort rule itself, so the table is loaded whole. Before a run, `python test_anonymise.py` in the code folder should pass (synthetic data only).

In [ ]:
derived_dir = open_picu_dir / "derived"
harmonised_dir = derived_dir / "harmonised"
assembled_dir = derived_dir / "assembled"
reports_dir = open_picu_dir / "reports"
out_root = derived_dir / "anonymised"
pd.set_option("display.max_colwidth", None)

print(AN.__file__)
print([hasattr(AN, n) for n in ("run", "preset", "diagnose", "diagnose_text", "text_vocabulary")])

static = pd.read_parquet(derived_dir / "static" / "pooled_static.parquet")
print(len(static), static["site"].value_counts().to_dict())

In [ ]:
def summarise(checks):
    """Counts of passes, warnings and failures, then every check that is not a pass."""
    print(checks["status"].value_counts().to_dict())
    return checks.loc[checks["status"] != "pass", ["check", "scope", "status", "n", "note"]]

## 1. Static table

The pooled static table (all sites, one row per stay). `run` applies the cohort rule itself.

In [ ]:
static = pd.read_parquet(derived_dir / "static" / "pooled_static.parquet")
print(len(static), static["site"].value_counts().to_dict())

## 2. Tier 1: pseudonymised, site kept, exact fields

Random stay and patient keys (one draw for all sites), hours since admission, no `t0` or patient ID, allow-listed columns, `ps_peep` renamed `ps_target_pressure`. Recorded text is kept as recorded after the note scan. Expect three warnings (long vocabularies in `value_text` at SMH and RBH); anything else that is not a pass needs a look.

In [ ]:
try:
    steps1, checks1 = AN.run(harmonised_dir, static, out_root, AN.TIER1, reports_dir=reports_dir)
except AN.CheckFailed as e:
    checks1 = e.checks
    raise
summarise(checks1)

## 3. Tier 2: pooled, no site, coarsened

One pooled table per group, no `site` column. Static fields coarsened and locally suppressed across all sites (k = 10), `split_group` instead of a patient key, five one-site concepts dropped, every GOSH drug row with a number and no unit dropped, rare text mapped to `rare`, `source` kept only for multi-source concepts, `pim` dropped. Nothing but passes is expected.

In [ ]:
try:
    steps2, checks2 = AN.run(harmonised_dir, static, out_root, AN.TIER2, reports_dir=reports_dir)
except AN.CheckFailed as e:
    checks2 = e.checks
    raise
summarise(checks2)

## 4. What the run changed

Rows changed by each rule, per file. `source_blanked` is left out (it blanks a constant index and loses no information). `drop_concept` is the five one-site concepts, `drop_drug_rule` the GOSH rows with a number and no unit, `text_note_nulled` and `raw_note_nulled` the note scan, `text_rare_mapped` the rare-string rule.

In [ ]:
rules = ["drop_concept", "drop_drug_rule", "drop_not_cohort", "text_note_nulled", "text_rare_mapped", "raw_note_nulled", "raw_rare_nulled", "time_rounded_rows"]
steps2[steps2["item"].isin(rules)].pivot_table(index="step", columns="item", values="n", aggfunc="sum", fill_value=0)

In [ ]:
print(checks2.loc[checks2["check"] == "small cells", "note"].iloc[0])
steps2[steps2["step"].str.startswith("static") | steps2["item"].astype(str).str.contains("too short")]

## 5. Diagnostics (run only when a check warns or fails)

Counts only, never values. `diagnose` lists, per string column, strings that equal or contain an original ID of any site (or are 32-character hex) and the rows they affect; it also reports rows far from admission. `diagnose_text` profiles the string columns. Use the site and group named in the failed check (for example `SMH_vent` is site "SMH", group "vent"). The original IDs under 5 characters are not matched inside text; the second cell shows where they are.

In [ ]:
site, group = "GOSH", "blood_gas"
ids_t, hours_t, by_concept = AN.diagnose(harmonised_dir, static, AN.TIER1, site, group)
display(ids_t, hours_t, by_concept)

In [ ]:
cols_t, concepts_t = AN.diagnose_text(harmonised_dir, static, AN.TIER1, "SMH", "vent")
display(cols_t, concepts_t)

In [ ]:
co = AN.cohort_static(static)
short = lambda s: s.dropna().astype(str).str.len() < AN.MIN_ID_LEN
pd.DataFrame({"short_stay_keys": co.groupby("site")["stay_key"].apply(lambda s: int(short(s).sum())),
              "short_patient_ids": co.groupby("site")["patient_id"].apply(lambda s: int(short(s).sum())),
              "stays": co.groupby("site").size()})

### Which condition of the note scan fired

The scan blanks `value_text` that is longer than 40 characters, has more than three words of three or more letters, contains a date, or has a run of seven or more digits. Per site, file and concept: rows and distinct strings hit, and which condition fired. If nearly all hits are `many_words`, the scan is blanking legitimate multi-word entries (vent mode, airway device) and the word rule is too strict for those concepts.

In [ ]:
rows = []
for site, grp in (("GOSH", "vent"), ("RBH", "vent"), ("SMH", "vent"), ("GOSH", "vitals"), ("RBH", "vitals"), ("SMH", "vitals")):
    r = pd.read_parquet(harmonised_dir / f"{site}_{grp}.parquet", columns=["concept", "value_text"]).dropna()
    t = r["value_text"].astype("string")
    f = pd.DataFrame({"concept": r["concept"].to_numpy(), "value_text": t.to_numpy(),
                      "too_long": (t.str.len() > 40).to_numpy(), "many_words": (t.str.count(r"[A-Za-z]{3,}") > 3).to_numpy(),
                      "date": t.str.contains(AN.DATE_RE, regex=True).to_numpy(), "long_digits": t.str.contains(r"\d{7,}", regex=True).to_numpy()})
    f["any"] = f[["too_long", "many_words", "date", "long_digits"]].any(axis=1)
    hit = f[f["any"]]
    if len(hit):
        s = hit.groupby("concept").agg(rows=("any", "size"), distinct=("value_text", "nunique"), too_long=("too_long", "sum"),
                                       many_words=("many_words", "sum"), date=("date", "sum"), long_digits=("long_digits", "sum"))
        s["rows_in_concept"] = f.groupby("concept").size().reindex(s.index)
        rows.append(s.assign(site=site, group=grp).reset_index())
pd.concat(rows, ignore_index=True)

## 6. Text vocabularies

How much of each recorded text vocabulary (`value_text`: vent mode, airway device, rhythm, pupils) is rare, per site and concept. Rare means seen in fewer than 10 stays; Tier 2 maps those strings to `rare`. A concept with many rare strings but few rare rows loses little; the share of stays touched shows how widely the tail is spread.

In [ ]:
vocab = AN.text_vocabulary(harmonised_dir)
vocab.sort_values("share_rows_rare", ascending=False)

## 7. Checks on the Tier 2 files

The row order must not reveal the site: walking down the sorted Tier 2 file, neighbouring stays should be at different sites about 0.6 of the time. A value near 0 would mean the file is blocked by site. The cell uses the private key map and prints one number.

Then the unit rule: GOSH rows with a number and no unit should be gone. The remaining count is the other sites' rows without a unit (pooled, so it cannot be split by site).

In [ ]:
km = pd.read_parquet(out_root / "_private" / "tier2_key_map.parquet").set_index("new_stay_key")["site"]
v = pd.read_parquet(out_root / "tier2" / "vitals.parquet", columns=["stay_key"])
order = km.reindex(v["stay_key"].drop_duplicates()).to_numpy()
shares = km.value_counts(normalize=True)
print("expected for a random order:", round(float(1 - (shares ** 2).sum()), 3))
print("share of neighbouring stays at different sites:", round(float((order[1:] != order[:-1]).mean()), 3))

In [ ]:
d2 = pd.read_parquet(out_root / "tier2" / "drugs.parquet", columns=["concept", "value", "unit"])
print("rows with a number and no unit in the Tier 2 drug file:", int((d2["value"].notna() & (d2["unit"].isna() | (d2["unit"].astype("string").str.strip() == ""))).sum()))

## 8. Figures

The staircase of stays in small cells after each blanked field (static table, all sites together). Check the figure with the output checker before it leaves the SDE.

In [ ]:
rep = steps2[steps2["step"] == "static: local suppression"].rename(columns={"item": "field", "n": "values_suppressed"})[["field", "values_suppressed"]]
R.plot_suppression(rep, int(steps2.loc[steps2["step"] == "static: rows", "n"].iloc[0]),
                   path=reports_dir / "tier2_suppression.png", title="Tier 2: stays in small cells after each blanked field");

## 9. Other levels (privacy and utility study)

A level is a preset with fields changed; give each its own `name` and `seed`. Presets: `tier2_pim` (PIM released as ten bands), `tier2_hourly` (time rounded to whole hours). Nothing else in the code changes. Uncomment to run.

In [ ]:
cfg = AN.preset("tier2", name="tier2_k20", k=20, seed=5, max_small_share=None)
steps3, checks3 = AN.run(harmonised_dir, static, out_root, cfg, reports_dir=reports_dir)
summarise(checks3)

### Tier 2 without ethnicity

Ethnicity left out of the static table, its source column, the suppression and the small-cell check. Seed 6, so its keys are unrelated to the other levels. Run it, then compare it with Tier 2 in `level_report.ipynb`.

In [ ]:
steps4, checks4 = AN.run(harmonised_dir, static, out_root, AN.PRESETS["tier2_no_ethnicity"], reports_dir=reports_dir)
summarise(checks4)

### Experiment: shared concepts only

Drops every concept recorded at fewer than three sites (the 10 one-site and 10 two-site concepts from the coverage table), on top of the existing one-site drops. This is an upper bound on how far removing concepts can reduce site inference, and the cost in content (urine output, IV intake, CRT, pupils) is what the privacy/utility comparison has to weigh.

In [ ]:
FEW_SITES = ("bg_venous_sat", "avpu", "bg_cvo2", "ett_cuff", "ett_depth", "ett_route", "gcs", "inhaled_no", "plateau_pressure", "temp_peripheral",
             "bg_hct", "bg_sample_type", "chloride", "crt", "cvp", "insp_time", "intake_iv", "pupil_reaction", "pupil_size", "urine_output")
cfg = AN.preset("tier2_no_ethnicity", name="tier2_shared", drop_concepts=AN.ONE_SITE_DROP + FEW_SITES)
steps, checks = AN.run(harmonised_dir, static, out_root, cfg, reports_dir=reports_dir)
summarise(checks)

## Check: do released `source` values name a site?

At Tier 2 the `source` column of the fluid files holds names such as `rbh_io`, `smh_io`, `gosh_diff` and `gosh_caboodle` for concepts with more than one source at some site. In a pooled file those names give the site away. This cell lists, for the levels already written, every distinct `source` value in every released file that contains a site or hospital name. An empty table means nothing leaks. The new `source labels` check does the same inside a run, but the levels written before it was added were never tested.

In [ ]:
import re
rows = []
for lvl in ["tier2", "tier2_no_ethnicity", "tier2_shared"]:
    for f in sorted((out_root / lvl).glob("*.parquet")):
        if f.stem == "static":
            continue
        cols = pd.read_parquet(f).columns
        if "source" not in cols:
            continue
        s = pd.read_parquet(f, columns=["source"])["source"].dropna().astype(str)
        for v, n in s.value_counts().items():
            if re.search(AN.SITE_WORDS, v):
                rows.append({"level": lvl, "file": f.name, "source": v, "rows": int(n)})
pd.DataFrame(rows)

### Level for comparison: Tier 2 without ethnicity, caboodle fluids kept

The default Tier 2 now drops the GOSH Epic-era fluid rows (`gosh_caboodle`). To measure what that costs, this level is the same as `tier2_no_ethnicity` but keeps them. `source_mode="none"` leaves the source column out, because with the caboodle rows kept the source names would give the site away and the `source labels` check would stop the run. Run `tier2_no_ethnicity` again first (it now drops the caboodle rows by default); then compare the two in `level_report.ipynb`, section 4.

In [ ]:
cfg = AN.preset("tier2_no_ethnicity", name="tier2_with_caboodle", drop_gosh_caboodle=False, source_mode="none", seed=11)
steps, checks = AN.run(harmonised_dir, static, out_root, cfg, reports_dir=reports_dir)
summarise(checks)

### Experiment: GOSH before the Epic switch only

`tier2_no_ethnicity` with every GOSH stay whose t0 is on or after 2019-04-20 left out (about 3,900 stays). It removes the Epic-era gaps (blood gas, ventilator mode, tidal volume, I:E ratio, diastolic pressures) and the incomplete caboodle fluids. The comparison in `level_report.ipynb` shows what this does to site inference. The baseline also changes, because GOSH is a smaller share, so compare the gain over the baseline, not the raw accuracy.

In [ ]:
AN.Config(gosh_epic_cutoff="2019-04-20")   # raises TypeError if the old class is still loaded

In [ ]:
cfg = AN.preset("tier2_no_ethnicity", name="tier2_pre_epic", gosh_epic_cutoff="2019-04-20", seed=12, max_small_share=None)
steps, checks = AN.run(harmonised_dir, static, out_root, cfg, reports_dir=reports_dir)
summarise(checks)

In [ ]:
display(checks[checks["check"] == "small cells"][["n", "note"]].style.set_properties(**{"white-space": "pre-wrap"}))
display(steps[steps["item"].str.contains("GOSH stays left out|^stays$", regex=True)])